# 12 因果推論 — 練習

用松柏護理之家退伍軍人症資料練習因果推論概念。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import statsmodels.formula.api as smf

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## 題目 1：水療暴露的歸因風險

1. 計算水療使用者（`hydrotherapy_use == 1`）與非使用者的侵襲率
2. 計算 AR（Attributable Risk）和 PAR（Population Attributable Risk）
3. 與淋浴暴露的 AR/PAR 比較，哪個暴露的貢獻更大？
4. 以因果推論的角度解讀：AR 代表什麼？需要什麼前提？

In [ ]:
# TODO: 水療使用者 vs 非使用者侵襲率
# TODO: AR 和 PAR
# TODO: 與淋浴暴露比較

## 題目 2：改變 DiD 介入日期

1. 將介入日從 1/25 改成 1/22
2. 介入組同樣是 2-3F B翼，對照組是其餘區域
3. 建立面板資料並執行 DiD 迴歸
4. `treated:post` 的係數和 p-value 有何變化？
5. 為什麼改變介入日期會影響結果？

In [ ]:
# TODO: 介入日改成 1/22
# TODO: 建立面板資料
# TODO: DiD 迴歸
# TODO: 比較結果

## 題目 3（挑戰題）：碰撞因子偏誤的實證

DAG 告訴我們 `hospitalized` 是碰撞因子（← severity, ← infection）。
如果只分析住院者，會產生假性關聯。

1. 計算全體的 shower_use → infected 的 RR
2. 只取住院者（`hospitalized == 1`），再算一次 RR
3. 兩個 RR 是否不同？為什麼？
4. 用碰撞因子的概念解釋差異

In [ ]:
# TODO: 全體 RR
# TODO: 只取住院者的 RR
# TODO: 比較差異
# TODO: 碰撞因子解讀

## 題目 4：疫苗接種政策的 DiD 評估（疫苗政策情境）

部分行政區推行加強接種活動，用差異中之差異（DiD）評估政策效果。

1. 用 2×2 組平均手算 DiD
2. 用 `smf.ols("incidence ~ treated * post")` 的交互項估計 DiD 並比較
3. 解讀交互項是否接近真值，並說明 DiD 如何去除共同時間趨勢

In [ ]:
# 疫苗接種率政策 DiD：部分行政區推行加強接種活動（treated），比較政策前後發生率
rng = np.random.default_rng(1204)
_rows = []
TRUE_EFFECT = -8.0   # 政策真正讓發生率下降 8/10萬
for dz in range(200):
    treated = 1 if dz < 100 else 0
    base = rng.normal(45, 6)
    for post in (0, 1):
        inc = base - 3 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 4)
        _rows.append({"district": dz, "treated": treated, "post": post, "incidence": inc})
vax = pd.DataFrame(_rows)
print(f"DiD 資料：{vax['district'].nunique()} 區 × 2 期，真值效果 = {TRUE_EFFECT}/10萬")

# TODO: 用 2x2 group means 手算 DiD =（treated 後-前）-（control 後-前）
# TODO: 用 smf.ols("incidence ~ treated * post", vax).fit() 取交互項係數，與手算比較
# TODO: 解讀交互項係數是否接近真值 -8，並說明 DiD 為何能去除共同時間趨勢

## 題目 5：口罩政策的 DiD 評估（口罩政策情境）

部分縣市實施口罩令，outcome 為每週病例成長率。

1. 用 `smf.ols("growth ~ treated * post")` 估計 DiD
2. 解讀口罩令的因果效果方向與大小

In [ ]:
# 口罩政策 DiD：部分縣市實施口罩令（treated），outcome 為每週病例成長率
rng = np.random.default_rng(1205)
_rows = []
TRUE_EFFECT = -0.18   # 口罩令讓成長率下降 0.18
for reg in range(160):
    treated = 1 if reg < 80 else 0
    base = rng.normal(0.30, 0.05)
    for post in (0, 1):
        g = base - 0.05 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 0.04)
        _rows.append({"region": reg, "treated": treated, "post": post, "growth": g})
mask = pd.DataFrame(_rows)
print(f"DiD 資料：{mask['region'].nunique()} 縣市 × 2 期，真值效果 = {TRUE_EFFECT}")

# TODO: 用 smf.ols("growth ~ treated * post", mask) 估計 DiD（交互項）
# TODO: 解讀口罩令的因果效果方向與大小

## 題目 6：吸菸與疾病——年齡干擾（吸菸情境）

年齡是吸菸與疾病的共同原因（干擾因子）。

1. 算 crude OR（`smf.logit("disease ~ smoke")`）
2. 算 adjusted OR（加入 `age`）
3. 比較 crude 與 adjusted 的 smoke 係數，解讀 age 的干擾方向

In [ ]:
# 吸菸與疾病：年齡是干擾因子（年長者較常吸菸、也較易生病）
rng = np.random.default_rng(1206)
n = 3000
age = rng.integers(20, 80, n)
smoke = rng.binomial(1, 1 / (1 + np.exp(-(-2.5 + 0.05 * age))))
logit = -4.5 + 0.05 * age + 0.8 * smoke     # smoke 真實 log-OR = 0.8
disease = rng.binomial(1, 1 / (1 + np.exp(-logit)))
dat = pd.DataFrame({"age": age, "smoke": smoke, "disease": disease})
print(f"n={n}，吸菸率={smoke.mean():.1%}，疾病率={disease.mean():.1%}（smoke 真實 log-OR=0.8）")

# TODO: 算 crude OR（只看 smoke vs disease，用 smf.logit("disease ~ smoke")）
# TODO: 算 adjusted OR（加入 age：smf.logit("disease ~ smoke + age")）
# TODO: 比較 crude 與 adjusted 的 smoke 係數，解讀 age 的干擾方向

## 題目 7：COVID-19 治療的傾向分數配對（COVID-19 情境）

病情越重越可能被治療（confounding by indication），用傾向分數配對估計治療效果。

1. 先看 naive 死亡率差為何偏誤
2. 用邏輯斯迴歸估計傾向分數（severity → treated）
3. 用 `NearestNeighbors` 對每個治療個案配對最近的對照
4. 算配對後 ATT，與真值 −0.15 比較

In [ ]:
# COVID-19 治療的傾向分數配對：病情越重越可能被治療（干擾），但治療其實有益
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import NearestNeighbors
rng = np.random.default_rng(1207)
n = 2000
severity = rng.uniform(0, 1, n)
treated = rng.binomial(1, 0.15 + 0.7 * severity)  # 越重越常被治療（confounding by indication；保留重疊）
TRUE_EFFECT = -0.15                              # 治療真正讓死亡率下降 0.15
death_p = (0.10 + 0.75 * severity + TRUE_EFFECT * treated).clip(0.01, 0.99)
death = rng.binomial(1, death_p)
cov = pd.DataFrame({"severity": severity, "treated": treated, "death": death})
print(f"n={n}，治療比例={treated.mean():.1%}，真值治療效果(死亡率差)={TRUE_EFFECT}")

# TODO: 先算 naive 死亡率差（treated - control），看它為何偏誤（甚至看起來有害）
# TODO: 用 LogisticRegression(severity → treated) 估計傾向分數
# TODO: 對每個 treated 個案用 NearestNeighbors 找傾向分數最近的 control 配對
# TODO: 算配對後的平均死亡率差（ATT），與真值 -0.15 比較

## 題目 8（挑戰題）：工具變數 IV / 2SLS（因果識別情境）

暴露受未觀測干擾 U 影響（內生），用工具變數 Z 做兩階段最小平方。

1. naive OLS：`smf.ols("outcome ~ exposure")` → 看它如何被 U 高估
2. 2SLS：第一階段 `exposure ~ Z` 取 fitted，第二階段 `outcome ~ exp_hat`
3. 比較 naive 與 IV 係數，說明 IV 為何能回復真值 1.5、Z 需滿足哪三個條件

In [ ]:
# 工具變數 IV：暴露受未觀測干擾 U 影響（內生），Z 為工具變數（挑戰題）
rng = np.random.default_rng(1208)
n = 3000
U = rng.normal(0, 1, n)                 # 未觀測干擾
Z = rng.normal(0, 1, n)                 # 工具變數：影響暴露、不直接影響結果
exposure = 0.6 * Z + 0.7 * U + rng.normal(0, 1, n)
TRUE_EFFECT = 1.5
outcome = TRUE_EFFECT * exposure + 1.2 * U + rng.normal(0, 1, n)
iv = pd.DataFrame({"Z": Z, "exposure": exposure, "outcome": outcome})
print(f"n={n}，暴露真實因果效果 = {TRUE_EFFECT}（naive OLS 會因 U 而高估）")

# TODO: naive OLS：smf.ols("outcome ~ exposure", iv) → 看係數如何被 U 高估
# TODO: 兩階段最小平方 2SLS：
#        stage1 = smf.ols("exposure ~ Z", iv).fit(); iv["exp_hat"]=stage1.fittedvalues
#        stage2 = smf.ols("outcome ~ exp_hat", iv).fit()
# TODO: 比較 naive 與 IV 的係數，說明為何 IV 能回復真值 1.5、以及 Z 需滿足什麼條件

## 題目 9：食物中毒的 AR/PAR 與清消措施的 DiD

某國小營養午餐爆發疑似食物中毒事件（本題資料為教學用合成情境，並非真實案例）。衛生單位懷疑當日供應的「涼拌小黃瓜」為可疑污染來源，稽核後要求團膳廠商加強清消措施。

**第一部分：AR / PAR（歸因風險）**

1. 用下方 2×2 表的 `a, b, c, d` 計算「吃了涼拌小黃瓜」與「沒吃」兩組的侵襲率
2. 計算風險比（RR）與歸因風險（AR，risk difference）
3. 用 Levin 公式計算族群歸因風險百分比（PAF）：`Pe * (RR - 1) / (1 + Pe * (RR - 1))`，其中 Pe 為全體中「有吃該菜色」的比例
4. 解讀 PAF：如果把這道菜從菜單中移除，理論上能減少多少比例的病例？

**第二部分：DiD（清消措施效果）**

5. 用下方面板資料，以 `smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")` 估計清消措施的介入效果（HC3 穩健標準誤）
6. `treated:post` 交互項代表什麼？和資料中設定的真值效果相比如何？
7. DiD 方法能成立的關鍵前提是什麼（平行趨勢假設）？如果介入前 treated 組和 control 組的病例趨勢本來就不平行，結果會如何被扭曲？

In [ ]:
# 食物中毒 2x2：吃了涼拌小黃瓜 vs 沒吃 × 發病 vs 未發病（教學用合成數據，非真實案例）
a, b = 90, 30    # 吃了嫌疑菜色：發病 / 未發病
c, d = 20, 180   # 沒吃嫌疑菜色：發病 / 未發病
print(f"吃了嫌疑菜色：{a + b} 人（發病 {a}，未發病 {b}）")
print(f"沒吃嫌疑菜色：{c + d} 人（發病 {c}，未發病 {d}）")

# 清消措施 DiD 面板：部分學校加強清消（treated），比較措施前後每日通報病例數（教學用合成數據）
rng = np.random.default_rng(1209)
_rows = []
TRUE_EFFECT = -4.0   # 加強清消措施讓每日通報病例數平均減少 4 例
for school in range(120):
    treated = 1 if school < 60 else 0
    base = rng.normal(10, 2)
    for post in (0, 1):
        cases = base - 1 * post + TRUE_EFFECT * (treated * post) + rng.normal(0, 1.5)
        _rows.append({"school": school, "treated": treated, "post": post, "cases": cases})
food = pd.DataFrame(_rows)
print(f"DiD 資料：{food['school'].nunique()} 校 × 2 期，真值效果 = {TRUE_EFFECT} 例/日")

In [ ]:
# TODO: 計算吃了嫌疑菜色 vs 沒吃兩組的侵襲率（attack rate）
# risk_exp = a / (a + b)
# risk_unexp = c / (c + d)

# TODO: 計算風險比 RR 和歸因風險 AR（risk difference）
# RR = risk_exp / risk_unexp
# AR = risk_exp - risk_unexp

# TODO: 計算 Pe（全體中「有吃該菜色」的比例），再用 Levin 公式算 PAF（族群歸因風險百分比）
# Pe = (a + b) / (a + b + c + d)
# PAF = Pe * (RR - 1) / (1 + Pe * (RR - 1))

# TODO: 解讀 PAF ——「移除這道菜理論上能減少的病例比例」

# TODO: 用 food 面板資料 fit DiD 迴歸
# fit = smf.ols("cases ~ treated + post + treated:post", data=food).fit(cov_type="HC3")

# TODO: 取出 treated:post 交互項係數與信賴區間，和真值效果比較

# TODO: 說明平行趨勢假設，若介入前兩組趨勢不平行，DiD 估計會如何被扭曲